# LangChain 실험 플레이그라운드

밈/신조어 키워드에 대해 질문 -> 벡터 검색(dense/sparse/융합) -> 프롬프트 조립 -> LLM 답변까지, 각 셀을 독립적으로 재실행하며 실험하기 위한 노트북입니다.

아래 셀들을 순서대로 한 번 실행한 뒤, 3번(질문) ~ 7번(LLM 호출) 셀만 값을 바꿔가며 반복 재실행하면 됩니다.

## 1. 환경 설정

**이 셀이 하는 일**: OS 환경변수, 인코딩, import를 준비합니다.

**바꿀 것**: 없음 — 노트북을 열면 한 번만 실행하면 됩니다.

**주의**: `.env`의 `MONGODB_URI`/`QDRANT_HOST`는 `mongo`/`qdrant`라는 docker-compose 내부 네트워크 호스트명을 가리킵니다. 이 이름은 `mimori-flask` 컨테이너 안에서만 resolve되므로, 이 노트북의 커널도 그 컨테이너 안에서 실행하거나(예: VS Code Dev Containers, `docker exec`로 jupyter 실행), 또는 `.env`를 로컬 실행용으로 `MONGODB_URI=mongodb://localhost:27017`, `QDRANT_HOST=localhost`로 바꿔서 실행해야 합니다. 그렇지 않으면 2번 셀에서 `ServerSelectionTimeoutError`가 발생합니다.

In [ ]:
import sys
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

# 이 노트북은 analysis/ 안에 있으므로 커널 작업 디렉토리는 analysis/ 입니다.
# 프로젝트 루트(analysis/의 상위 폴더)를 import 경로에 추가합니다.
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

if sys.platform == "win32":
    try:
        sys.stdin.reconfigure(encoding="utf-8")
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except AttributeError:
        pass

from analysis.pipeline import list_analyzable_keywords
from analysis.rag_pipeline import (
    search_relevant_chunks,
    search_dense_only,
    search_sparse_only,
)
from embedding.encoder import encode_batch
from config.config_cilent import NIM_KEY
from langchain_nvidia_ai_endpoints import ChatNVIDIA

print("설정 완료")

## 2. 키워드 선택

**이 셀이 하는 일**: MongoDB에서 임베딩이 끝난 밈 키워드 목록을 가져와 보여줍니다.

**실험하려면**: 출력된 목록 중 하나를 골라 `KEYWORD` 변수에 문자열로 직접 대입한 뒤 재실행하세요.

In [ ]:
keywords = list_analyzable_keywords()
print(f"분석 가능한 키워드 {len(keywords)}개:")
for kw in keywords:
    print(f"  - {kw}")

KEYWORD = keywords[0] if keywords else ""
print(f"\n현재 선택된 KEYWORD = {KEYWORD!r}")

## 2.5. z-score / 유행 상태 확인

**이 셀이 하는 일**: 선택한 키워드(`KEYWORD`)의 최근 검색량 기반 유행 상태를 네이버 데이터랩에서 조회합니다.

**실험하려면**: 아래 코드 셀의 `INCLUDE_TREND`를 `True`/`False`로 바꾼 뒤, 이 셀을 먼저 재실행하고 "6. 프롬프트 작성" 셀을 실행해야 반영됩니다(`INCLUDE_TREND`는 이 셀에서 정의되므로, 값만 바꾸고 이 셀을 다시 실행하지 않으면 이전 값이 그대로 남아있습니다).

In [ ]:
from trend.trend_service import format_trend_context

trend_info = format_trend_context(KEYWORD)
if trend_info:
    print(trend_info)
else:
    print("z-score/유행 상태를 가져오지 못했습니다 (NAVER API 키 미설정이거나 데이터 없음).")

INCLUDE_TREND = True

## 3. 질문 입력 + 임베딩

**이 셀이 하는 일**: 자유 텍스트 질문을 BGE-M3로 dense 벡터 + sparse(lexical) 벡터로 변환합니다.

**실험하려면**: `QUESTION` 문자열만 바꾸면 됩니다. 질문이 바뀌면 검색되는 청크가 달라지므로, 이 셀부터 다시 실행해야 아래 결과에 반영됩니다. (임베딩 모델은 재실행마다 새로 로드하지 않도록 그대로 유지합니다 — LLM 호출은 NVIDIA API라 로컬 GPU와 충돌하지 않습니다.)

In [ ]:
QUESTION = "이 밈은 왜 유행했나요?"

dense_vecs, lexical_weights = encode_batch([QUESTION])
dense_vec, sparse = dense_vecs[0], lexical_weights[0]

print(f"QUESTION = {QUESTION!r}")
print(f"dense_vec 길이 = {len(dense_vec)}")
print(f"sparse 항목 수 = {len(sparse)}")

## 4. 검색 파라미터 + 실행

**이 셀이 하는 일**: 같은 질문 벡터로 dense만 / sparse만 / RRF 융합, 세 가지 방식으로 Qdrant에서 관련 청크를 검색합니다.

**실험하려면**: `TOP_K`를 늘리거나 줄여서 더 많은/적은 근거 청크를 가져와볼 수 있습니다.

In [ ]:
TOP_K = 5

dense_points = search_dense_only(KEYWORD, dense_vec, TOP_K)
sparse_points = search_sparse_only(KEYWORD, sparse, TOP_K)
fused_points = search_relevant_chunks(KEYWORD, dense_vec, sparse, TOP_K)

print(f"dense={len(dense_points)}, sparse={len(sparse_points)}, fused={len(fused_points)}")

## 5. 검색 결과 확인

**이 셀이 하는 일**: dense / sparse / 융합(RRF) 검색 결과를 나란히 비교 출력합니다(점수, 제목, 출처, 본문 일부). 벡터 검색이 실제로 어떤 데이터를 가져오는지 여기서 확인할 수 있습니다.

**실험하려면**: 이 셀 자체는 결과를 출력만 하므로 수정할 것이 없습니다 — 위 셀들(질문, TOP_K, 키워드)을 바꾸고 재실행해서 차이를 비교하세요.

In [ ]:
def _print_points(label, points):
    print(f"--- {label} ({len(points)}개) ---")
    for p in points:
        title = p.payload.get("title") or "제목 없음"
        url = p.payload.get("url") or "출처 없음"
        text = p.payload.get("text", "")
        print(f"  score={p.score:.4f} | {title} ({url})")
        print(f"  {text[:150]}")
    print()

_print_points("DENSE", dense_points)
_print_points("SPARSE", sparse_points)
_print_points("FUSED(RRF)", fused_points)

## 6. 프롬프트 작성

**이 셀이 하는 일**: 검색된 청크(`fused_points`)와 유행 상태 정보(`trend_info`, `INCLUDE_TREND`가 `True`일 때만)를 근거 자료로 넣어 LLM에게 실제로 전달할 프롬프트를 완성합니다.

**실험하려면**: `PROMPT_TEMPLATE` 문자열을 자유롭게 수정하세요(지시문 추가, 어조 변경, 출력 형식 지정 등). `{keyword}` / `{trend_info}` / `{context}` / `{question}` 네 자리표시자는 반드시 그대로 남겨둬야 합니다. `.format()`을 쓰므로, 프롬프트에 `{`/`}` 자체를 문자로 넣고 싶으면(예: JSON 출력 형식을 지시하는 경우) `{{`/`}}`로 이스케이프해야 합니다.

In [ ]:
PROMPT_TEMPLATE = """당신은 밈/신조어 분석 전문가입니다.

키워드: {keyword}

{trend_info}
자료:
{context}

질문: {question}

위 자료를 근거로 답변하세요. 자료에 없는 내용은 추측하지 말고 모른다고 답하세요.
"""

context = "\n\n---\n\n".join(
    f"[출처: {p.payload.get('title') or '제목 없음'} / {p.payload.get('url') or '출처 없음'}]\n{p.payload.get('text', '')}"
    for p in fused_points
)
prompt = PROMPT_TEMPLATE.format(
    keyword=KEYWORD,
    trend_info=(trend_info if INCLUDE_TREND else ""),
    context=context,
    question=QUESTION,
)
print(prompt)

## 7. LLM 호출

**이 셀이 하는 일**: 완성된 프롬프트를 NVIDIA API(ChatNVIDIA)로 보내고 답변을 받아 출력합니다.

**실험하려면**: `MODEL` / `TEMPERATURE` / `TOP_P` 값을 바꿔서, 같은 프롬프트에도 답변이 어떻게 달라지는지 비교해보세요.

In [ ]:
assert NIM_KEY, "NIM_KEY가 .env에 설정되어 있지 않습니다"

MODEL = "deepseek-ai/deepseek-v4-flash"
TEMPERATURE = 1
TOP_P = 0.95

llm_client = ChatNVIDIA(
    model=MODEL,
    api_key=NIM_KEY,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_completion_tokens=16384,
    timeout=6000,
)

response = llm_client.invoke([{"role": "user", "content": prompt}])
print(response.content)